<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Word error rate, step by step

**[Word error rate, step by step](https://learning.nextia-ai.com/courses/speech/m02/word-error-rate-step-by-step/)** · Speech AI: Transcribe, Understand and Speak · Module 2, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** count substitutions, deletions and insertions between a reference and a model transcript, compute the word error rate (WER) and the character error rate (CER) by hand and with code, check the result with `jiwer`, and see why WER can be over 100%.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key and no model download. |
| **You should know** | What a reference transcript is, from [Set up the speech project](https://learning.nextia-ai.com/courses/speech/m01/set-up-the-speech-project/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m02/word-error-rate-step-by-step/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M02-L01-word-error-rate-step-by-step/word-error-rate-step-by-step-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, jiwer 4.0.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2 jiwer==4.0.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile', 'jiwer']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 2 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "22fe4e147bf62b4ff6c3b6b4a1cc559b048e12a0c2b3c8098eb72982f2d9e695",
    "data/recordings/asr/whisper-tiny.jsonl": "d6c12c960a2bc99981f866d674830b7501c01f754ae0f11d8bacf1139aa7285b",
    "compare.py": "99a57df2715cd87e50ce2b967afd530067dc626149f960b32d38f683d39c905f",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Three kinds of mistake

Grace reads a transcript of a call: "i do want the refund please". The caller said: "i do not want a refund". The model made three kinds of mistake:

| Kind | What happened | Here |
|---|---|---|
| **Substitution** (S) | a word became another word | "a" became "the" |
| **Deletion** (D) | a word is missing | "not" is missing |
| **Insertion** (I) | a word was added | "please" was added |

**WER = (S + D + I) / N**, where N is the number of words in the reference. Here: (1 + 1 + 1) / 6 = 0.5. Half of the words are wrong, and the one that matters most, "not", is gone.

To count, you first **align** the two texts: you pair each reference word with a model word, so that the number of changes is as small as possible. The next cell does it with `cp.align`, the course's helper.

In [ ]:
reference_text = "i do not want a refund"
model_text = "i do want the refund please"
for op, ref_word, model_word in cp.align(reference_text.split(), model_text.split()):
    print(f"{op:4}  {str(ref_word):8}  {str(model_word):8}")

> **Check.** Each line is one step: `ok` (the same word), `sub`, `del` (only a reference word) or `ins` (only a model word).

The next cell counts the steps and computes the WER.

In [ ]:
c = cp.counts(reference_text, model_text)
print(c)
print("WER =", (c["sub"] + c["del"] + c["ins"]), "/", c["ref_words"], "=", round(cp.wer(reference_text, model_text), 3))

> **Check.** `sub` 1, `del` 1, `ins` 1 and `ref_words` 6: WER = 0.5.

## 2. How the alignment is found

The alignment is the **minimum edit distance** (Levenshtein distance) between the two lists of words. A table holds, in each cell, the fewest changes that turn the first `i` reference words into the first `j` model words. Each cell takes the smallest of three moves: from the left (an insertion), from above (a deletion) or from the diagonal (the same word, or a substitution).

Run the next cell. The number in the bottom-right corner is S + D + I.

In [ ]:
def edit_table(ref_words, hyp_words):
    d = np.zeros((len(ref_words) + 1, len(hyp_words) + 1), dtype=int)
    d[:, 0] = range(len(ref_words) + 1)
    d[0, :] = range(len(hyp_words) + 1)
    for i in range(1, len(ref_words) + 1):
        for j in range(1, len(hyp_words) + 1):
            same = ref_words[i - 1] == hyp_words[j - 1]
            d[i, j] = min(d[i - 1, j - 1] + (0 if same else 1), d[i - 1, j] + 1, d[i, j - 1] + 1)
    return d

table = edit_table(reference_text.split(), model_text.split())
print("        " + " ".join(f"{w[:6]:>6}" for w in ["-"] + model_text.split()))
for word, row in zip(["-"] + reference_text.split(), table):
    print(f"{word[:6]:>6}  " + " ".join(f"{v:>6}" for v in row))

## 3. jiwer agrees

`jiwer` is a common Python package for WER. It does not normalise for you (except spaces), so give it the normalised texts. Run the next cell.

In [ ]:
import jiwer
out = jiwer.process_words(reference_text, model_text)
print("jiwer: S", out.substitutions, "D", out.deletions, "I", out.insertions, "WER", out.wer)
print("helper WER", cp.wer(reference_text, model_text), "| jiwer CER", round(jiwer.cer(reference_text, model_text), 3),
      "| helper CER", round(cp.cer(reference_text, model_text), 3))

> **Check.** Both give a WER of 0.5 and the same CER.

## 4. A real call

Now a real model transcript: the course's recorded run of Whisper `tiny` on the phone version of call C-0056 (a damaged item; the caller says the order number in pairs). Both texts go through the normalisation guide first.

In [ ]:
calls = load_calls()
tiny = {}
for line in open("data/recordings/asr/whisper-tiny.jsonl", encoding="utf-8"):
    row = json.loads(line)
    if row["kind"] == "phone":
        tiny[row["call_id"]] = row["text"]
call = calls["C-0056"]
ref = cp.normalise(reference(call), "full", "en")
hyp = cp.normalise(tiny["C-0056"], "full", "en")
c = cp.counts(ref, hyp)
print(c, " WER", round(cp.wer(ref, hyp), 3))

In [ ]:
print(jiwer.process_words(ref, hyp).wer)
for op, r, h in cp.align(ref.split(), hyp.split()):
    if op != "ok":
        print(op, r, "->", h)

> **Check.** jiwer gives the same WER as the helper. Read the errors: which ones would change what Grace's team does?

## 5. Predict: can WER be over 100%?

> **Predict.** The caller says one word, "Yes.", and then stays silent. In the silence, the model writes "Yes. Thank you for watching." (a known habit of Whisper in silence; we wrote this example). What is the WER?

In [ ]:
ref_short = cp.normalise("Yes.")
hyp_long = cp.normalise("Yes. Thank you for watching.")
print(cp.counts(ref_short, hyp_long), "WER", cp.wer(ref_short, hyp_long))

> **Check.** 4 insertions for 1 reference word: WER = 4.0, so 400%. WER has no upper limit, because insertions have no limit. A WER over 1 often means that the model invented text in silence or music. Module 3 shows how to stop it.

## 6. Change one thing: one wrong letter

WER counts whole words. CER (character error rate) counts letters, with the same alignment on characters. Run the next cell.

> **Predict.** The model writes "teracotta" for "terracotta". Is the WER or the CER of this sentence larger?

In [ ]:
a = "the terracotta trio arrived broken"
b = "the teracotta trio arrived broken"
print("WER", cp.wer(a, b), " CER", round(cp.cer(a, b), 3))

> **Check.** WER 0.2 (one word in five), CER about 0.029 (one letter in 34 characters). CER is kinder to small spelling errors. It is useful for languages without spaces and for names; WER is the usual score for transcripts.

## 7. Failure case: reference and model the wrong way round

WER divides by the number of **reference** words. If you swap the two arguments, you get a different number, with no error message.

In [ ]:
print("right:", cp.wer(ref_short, hyp_long), "  swapped:", round(cp.wer(hyp_long, ref_short), 3))
print("jiwer right:", jiwer.wer(ref_short, hyp_long), "  swapped:", round(jiwer.wer(hyp_long, ref_short), 3))

> **Check.** For the "Yes." example: 4.0 the right way, 0.8 swapped. Swapped, the 4 extra words look like 4 deletions out of 5 words. **Fix:** always pass the reference first (`jiwer.wer(reference, hypothesis)`), and name your variables `ref` and `hyp`.

## 8. Your turn: WER and CER of call C-0196

> **Your turn.** Call C-0196 is a cancelled order on a VoIP line with street noise. Normalise its reference transcript and the recorded Whisper `tiny` transcript (`tiny["C-0196"]`) with the guide (`"full"`, the call's language). Then use `cp.align` to count the word steps that are not `ok`, and put the WER in `task_wer`. Do the same on the characters (`list(ref)`, spaces included) for `task_cer`.

In [ ]:
task_wer = ...
task_cer = ...

In [ ]:
expect('task_wer', task_wer, 0.0965, tolerance=0.001)

In [ ]:
expect('task_cer', task_cer, 0.0471, tolerance=0.001)

> **Check.** Your numbers should be the same as `cp.wer(ref, hyp)` and `jiwer.cer(ref, hyp)`. Try it.

## Next

You can count S, D and I, compute WER and CER, and check them with `jiwer`. The next lesson shows how much the score changes when only the normalisation changes. The [lesson page](https://learning.nextia-ai.com/courses/speech/m02/word-error-rate-step-by-step/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Normalise before you compare](https://learning.nextia-ai.com/courses/speech/m02/normalise-before-you-compare/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m02/word-error-rate-step-by-step/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.